# Opis projektu

Celem projektu jest implementacja gry logicznej Clobber polegającej na zbijaniu pionków przeciwnika. Warunek wygranej jest wykonanie ostatniego możliwego ruchu. Gra rozgrywa się na planszy o domyślnym rozmiarze 8x8, która jest podzielona na pola z naprzemiennie występującymi białymi i czarnymi pionkami. W grze bierze udział dwóch graczy sterowanych przez algorytm Minmax z rozszerzeniem o Alpha-Beta. Algorytm Minmax jest algorytmem przeszukiwania drzewa decyzyjnego, który znajduje optymalne ruchy w grach dwuosobowych. Rozszerzenie Alpha-Beta pozwala na przyspieszenie tego procesu poprzez eliminację niepotrzebnych gałęzi drzewa. Optymalność ruchu jest zapewniona przez wykorzystanie heurystyk, które oceniają stan planszy.

# Struktura projektu

Projekt został wykonany w języku Python i został podzielony na moduły logiczne:
* `src/algorithms` - moduł zawierający zaimplementowane algorytmy Minmax oraz rozszerzenie Alpha-Beta oraz heurystyki
* `src/enums` - moduł zawierający enumeracje wykorzystywane w projekcie, takie jak kolory pionków, stan gry czy kierunki ruchu
* `src/game` - moduł zawierający implementację planszy oraz pętli gry
* `src/player` - moduł przechowujący implementację graczy
* `src/utils` - moduł zawierający dodatkowe funkcjonalności, takie jak dekoratory

# Implementacja planszy

Plansza gry jest, reprezentowana przez klasę `Board`, zostałą zaimplementowana jako dwuwymiarowa tablica przechowująca informację o obecnym stanie pól, czy komórek reprezentowanych przez `CellState`. Stan pola może być pusty, biały lub czarny i jest wykorzystywany w celu obliczania możliwych ruchów. Klasa `Board` zawiera m.in metody obliczające możliwe ruchu, sprawdzające czy dany ruch jest poprawny oraz wykonujące ruch.

In [ ]:
from typing import Dict, List, Tuple, Optional
from src.enums.color import Color
from src.enums.move_direction import MoveDirection
from src.enums.cell_state import CellState
from src.game.move import Move

class Board:
    _board: List[List[int]]
    _n: int
    _m: int

    def __init__(self, n: int = 10, m: int = 10):
        self._n = n
        self._m = m
        self._board = self._create_board(n, m)

    @property
    def center(self) -> Tuple[int, int]:
        return ((self._n - 1) / 2, (self._m - 1) / 2)

    def _create_board(self, n: int, m: int) -> List[List[int]]:
        board = []
        for y in range(n):  
            row = []
            for x in range(m):  
                state = CellState.WHITE if (x + y) % 2 == 0 else CellState.BLACK
                row.append(state.value)
            board.append(row)
        return board

    def print_board(self) -> None:
        for row in self._board:
            print(" ".join(str(CellState(cell)) for cell in row))

    def get_cell(self, position: Tuple[int, int]) -> CellState:
        x, y = position
        if not self.is_within_bounds(x, y):
            return CellState.EMPTY
        return CellState(self._board[y][x])

    def set_cell(self, position: Tuple[int, int], state: CellState) -> None:
        x, y = position
        if state is None:
            state = CellState.EMPTY
        self._board[y][x] = state.value

    def get_copy(self) -> "Board":
        new_board = Board(self._n, self._m)
        new_board._board = [row[:] for row in self._board]
        return new_board

    def perform_move(self, move: Move) -> Optional[CellState]:
        start_x, start_y = move.x_start, move.y_start
        end_x, end_y = move.x_end, move.y_end

        start_cell = self.get_cell((start_x, start_y))
        captured_cell = self.get_cell((end_x, end_y))

        if captured_cell == CellState.EMPTY or start_cell == captured_cell:
            return None

        self.set_cell((end_x, end_y), start_cell)
        self.set_cell((start_x, start_y), CellState.EMPTY)
        return captured_cell

    def undo_move(self, move: Move, captured_cell: Optional[CellState]) -> None:
        start_x, start_y = move.x_start, move.y_start
        end_x, end_y = move.x_end, move.y_end
        end_cell = self.get_cell((end_x, end_y))

        self.set_cell((start_x, start_y), end_cell)
        self.set_cell(
            (end_x, end_y),
            captured_cell if captured_cell is not None else CellState.EMPTY,
        )

    def get_cells_with_color(self, color: Color) -> List[Tuple[int, int]]:
        cells = []
        for y in range(self._n):
            for x in range(self._m):
                if self._board[y][x] == color.value:
                    cells.append((x, y))
        return cells

    def get_all_cells(self) -> List[Tuple[int, int]]:
        return [(x, y) for y in range(self._n) for x in range(self._m)]

    def is_within_bounds(self, x: int, y: int) -> bool:
        return 0 <= x < self._m and 0 <= y < self._n

    def calculate_state(self) -> Tuple[int, int, str]:
        white_points = sum(row.count(Color.WHITE.value) for row in self._board)
        black_points = sum(row.count(Color.BLACK.value) for row in self._board)
        return white_points, black_points


    def calculate_possible_moves(self, color: Color) -> List[Move]:
        enemy_color = -color
        player_cells = self.get_cells_with_color(color)
        possible_moves = []

        for x, y in player_cells:
            for direction in MoveDirection:
                dx, dy = direction.value
                new_x, new_y = x + dx, y + dy

                if not self.is_within_bounds(new_x, new_y):
                    continue

                end_cell = self.get_cell((new_x, new_y))
                if end_cell.to_color() == enemy_color:
                    possible_moves.append(Move(0, x, y, new_x, new_y))
        return possible_moves

    def is_valid_move(self, move: Move, color: Color) -> bool:
        if not self.is_within_bounds(move.x_end, move.y_end):
            return False
        end_cell_color = self.get_cell((move.x_end, move.y_end)).to_color()
        return end_cell_color == -color

    def is_isolated(self, x: int, y: int) -> bool:
        color = self.get_cell((x, y)).to_color()
        if not color:
            raise ValueError("This cell is empty, it cannot be isolated")

        for dx, dy in MoveDirection.all():
            new_x, new_y = x + dx, y + dy
            if not self.is_within_bounds(new_x, new_y):
                continue

            if self.get_cell((new_x, new_y)).to_color() == -color:
                return False
        return True

    def total_cells(self) -> int:
        return len(self)

    def occupied_cells(self) -> int:
        return sum(
            1 for row in self._board for cell in row if cell != CellState.EMPTY.value
        )

    def pieces_count(self, color: Color) -> int:
        return sum(row.count(color.value) for row in self._board)

    def get_hash(self) -> int:
        return hash(tuple(tuple(row) for row in self._board))

    def __len__(self) -> int:
        return self._n * self._m


# Implementacja gracza

Gracz jest definiowany na podstaiwe koloru, wersji algorytmu Minmax (podstawowej lub z rozszerzeniem ALpha-Beta) i jej głebokości przeszukiwania. Dodatkowo jest podawana heurystyka wykorzystywana do oceny stanu planszy przez gracza. Gracz jest reprezentowany przez klasę `Player`, która zawiera metodę `choose_move`, która zwraca najlepszy ruch dla danego gracza. Metoda `acknowledge_outcome` jest wykorzystywana do informowania gracza o wyniku gry. Ta informacja jest wykorzystywana w przypadku uczących się heurystyk, które mogą dostosować swoje parametry np. wagi na podstawie wyniku.

In [ ]:
from typing import List, Tuple
from src.algorithms.heuristics import (
    Heuristic,
    IsolationHeuristic,
)
from src.algorithms.minmax import AlphaBetaMinmax, Minmax

from src.game.board import Board
from src.enums.color import Color
from src.game.move import Move


class Player:
    color: Color
    minmax: Minmax
    current_heuristic: Heuristic

    def __init__(
        self, color: Color, minmax: Minmax, heuristic: Heuristic | None = None
    ) -> None:
        if heuristic is None:
            heuristic = IsolationHeuristic()
        self.color = color
        self.minmax = minmax
        self.current_heuristic = heuristic

    def choose_move(self, board: Board) -> Move | None:
        if not self.minmax:
            return board.calculate_possible_moves(self.color)[0]
        return self.minmax.execute(board, self.color, self.current_heuristic)

    def __str__(self) -> str:
        return f"Player {self.color}"

    def __repr__(self) -> str:
        return f"Player(color={self.color}, minmax={self.minmax}, heuristic={self.current_heuristic})"

    def acknowledge_experiences(
        self,
        experiences: List[Tuple[Board, Color]],
        outcome: int,
    ) -> None:
        experiences = [
            (board, color, outcome)
            for board, color in experiences
            if color == self.color
        ]
        self.current_heuristic.apply_experiences(experiences)

    @staticmethod
    def create_default_players(
        minmax_cls=AlphaBetaMinmax,
        depth: int = 4,
        heuristic: Heuristic | None = None,
    ) -> Tuple["Player", "Player"]:
        if heuristic is None:
            heuristic = IsolationHeuristic

        white_player = Player(Color.WHITE, minmax_cls(depth), heuristic())
        black_player = Player(Color.BLACK, minmax_cls(depth), heuristic())
        return white_player, black_player


# Implementacja algorytmu Minmax oraz Alpha-Beta

Implementacja algorytmu Minmax oraz Alpha-Beta została zrealizowana w klasie `Minmax` oraz klasach dziedziczących. Zawierają one metodę `execute`, która przyjmuje takie parametry jak plansza, głębokość pozsukiwania, kolor gracza oraz stosowaną heurystykę, a sama implementacja algorytmu została zrealizowana w metodzie `_minmax`. Algorytm Minmax przeszukuje drzewo decyzyjne i zwraca najlepszy ruch dla danego gracza. Algorytm Alpha-Beta działa na tej samej zasadzie, ale dodatkowo wykorzystuje zmienne alpha i beta do przycinania gałęzi drzewa, które nie mogą wpłynąć na wynik końcowy. Dzięki temu algorytm Alpha-Beta jest bardziej efektywny i szybszy od algorytmu Minmax. Dodatkowo została zaimplementowana funkcjonalność zapamiętywania stanów planszy, aby uniknąć ponownego obliczania wartości dla tych samych stanów.

In [ ]:
from abc import ABC, abstractmethod
import random
from typing import Tuple

from src.enums.color import Color
from src.game.board import Board
from src.game.move import Move
from src.algorithms.heuristics import Heuristic


class Minmax(ABC):
    def __init__(self, depth: int = 2):
        self.depth = depth
        self.transposition_table = {}

    @abstractmethod
    def execute(
        self, board: Board, player_color: Color, heuristic: Heuristic | None = None
    ) -> Move | None:
        raise NotImplementedError("Subclasses should implement this method.")

    def _minmax(
        self,
        board: Board,
        depth: int,
        maximizing: bool,
        color: Color,
        self_color: Color,
        alpha: float = float("-inf"),
        beta: float = float("inf"),
        use_pruning: bool = False,
        heuristic: Heuristic | None = None,
    ) -> Tuple[float, Move | None]:
        board_hash = (board.get_hash(), color.value, maximizing)

        if board_hash in self.transposition_table:
            return self.transposition_table[board_hash]

        if depth == 0:
            value = heuristic.evaluate(board, self_color)
            self.transposition_table[board_hash] = (value, None)
            return value, None
        moves = board.calculate_possible_moves(color)
        if not moves:
            return float("-inf") if maximizing else float("+inf"), None

        best_move: Move = None
        best_eval = float("-inf") if maximizing else float("inf")

        for move in moves:
            captured = board.perform_move(move)

            eval_score, _ = self._minmax(
                board,
                depth - 1,
                not maximizing,
                -color,
                self_color,
                alpha,
                beta,
                use_pruning,
                heuristic,
            )

            board.undo_move(move, captured)
            move.score = eval_score

            if (maximizing and eval_score >= best_eval) or (
                not maximizing and eval_score <= best_eval
            ):
                best_eval = eval_score
                best_move = move

            if use_pruning:
                if maximizing:
                    alpha = max(alpha, eval_score)
                else:
                    beta = min(beta, eval_score)
                if beta <= alpha:
                    break

        self.transposition_table[board_hash] = (best_eval, best_move)
        return best_eval, best_move


class BaseMinmax(Minmax):
    def execute(
        self, board: Board, color: Color, heuristic: Heuristic | None = None
    ) -> Move | None:
        heuristic = heuristic
        self.transposition_table.clear()
        _, move = self._minmax(
            board, self.depth, True, color, color, heuristic=heuristic
        )
        return move


class AlphaBetaMinmax(Minmax):
    def execute(
        self, board: Board, color: Color, heuristic: Heuristic | None = None
    ) -> Move | None:
        heuristic = heuristic
        self.transposition_table.clear()
        _, move = self._minmax(
            board, self.depth, True, color, color, use_pruning=True, heuristic=heuristic
        )
        return move


# Implementacja heurystyk

Heurystyki, które zostały zaimplementowane w projekcie, to:
* `IsolationHeuristic` - ocenia stan planszy na podstawie izolacji pionków gracza i przeciwnika. Izolacja pionka oznacza, że ten pionek nie ma możliwości wykonania ruchu.
* `ControlHeuristic` - ocenia stan planszy na podstawie kontroli wokół środka planszy.
* `ConnectivityHeuristic` - ocenia stan planszy na podstaiwe występowania grup pionków. Im większa grupa połączonych pionkó, tym lepsza ocena.
* `EdgePositionHeuristic` - ocenia stan planszy na podstawie zajmowania krawędzi i rogów planszy. Im więcej pionków występujących na krawędziach tym gorsza ocena.
* `RandomHeuristic` - ocenia stan planszy losowo.
* `WeightedHeuristic` - ocenia stan planszy na podstawie wag przypisanych do wybranych heurystyk.
* `RLHeuristic` - podobnie jak WeightedHeuristic, stan jest oceniany na podstawie wag przypisanych do wybranych heurystyk. Różni się natomiast tym, że wagi są obliczane na podstawie algorytmu Q-learning. Algorytm Q-learning jest algorytmem uczenia się ze wzmocnieniem, który uczy się optymalnych wag na podstawie doświadczenia. Aby algorytm mógł się uczyć, wymagane jest wiele iteracji gry - rund.

In [ ]:
from abc import ABC, abstractmethod
import random
from typing import List, Tuple
from src.enums.move_direction import MoveDirection
from src.game.board import Board
from src.enums.color import Color
from math import tanh


class Heuristic(ABC):
    def __init__(self):
        self._cache = {}

    @abstractmethod
    def _evaluate(self, board: Board, color: Color) -> float:
        pass

    def evaluate(self, board: Board, color: Color) -> float:
        value = self._evaluate(board, color)
        return value

    def clear_cache(self):
        self._cache.clear()

    def apply_experiences(self, experiences: List[Tuple[Board, Color, float]]) -> None:
        pass


class IsolationHeuristic(Heuristic):
    def _evaluate(self, board: Board, color: Color) -> float:
        player_score = self._calculate_score(board, color)
        opponent_score = self._calculate_score(board, -color)
        return player_score - opponent_score

    def _calculate_score(self, board: Board, color: Color) -> int:
        cells = board.get_cells_with_color(color)
        total = len(cells)
        isolated = sum(1 for x, y in cells if board.is_isolated(x, y))
        return total - isolated


class ControlHeuristic(Heuristic):
    def _evaluate(self, board, color):
        player_score = self._calculate_score(board, color)
        opponent_score = self._calculate_score(board, -color)
        return player_score - opponent_score

    def _calculate_score(self, board: Board, color: Color) -> float:
        cells: List[Tuple[int, int]] = board.get_cells_with_color(color)
        score = sum(
            map(lambda cell: self._calculate_distance_score(board, cell), cells)
        )
        return score

    def _calculate_distance_score(
        self, board: Board, position: Tuple[int, int]
    ) -> float:
        x, y = position
        cx, cy = board.center
        return 1 / (1 + (x - cx) ** 2 + (y - cy) ** 2)


class ConnectivityHeuristic(Heuristic):
    def _evaluate(self, board: Board, color: Color) -> float:
        player_score = self._calculate_connectivity(board, color)
        opponent_score = self._calculate_connectivity(board, -color)
        return player_score - opponent_score

    def _calculate_connectivity(self, board: Board, color: Color) -> int:
        cells = board.get_cells_with_color(color)
        visited = set()
        total = 0

        for x, y in cells:
            if (x, y) not in visited:
                group_size = self._flood_fill(board, x, y, color, visited)
                total += group_size * group_size
        return total

    def _flood_fill(
        self, board: Board, x: int, y: int, color: Color, visited: set
    ) -> int:
        stack = [(x, y)]
        count = 0

        while stack:
            cx, cy = stack.pop()
            if (cx, cy) in visited or not board.is_within_bounds(cx, cy):
                continue

            if board.get_cell((cx, cy)).to_color() == color:
                visited.add((cx, cy))
                count += 1
                for dx, dy in MoveDirection.all():
                    stack.append((cx + dx, cy + dy))
        return count


class EdgePositionHeuristic(Heuristic):
    def _evaluate(self, board: Board, color: Color) -> float:
        player_penalty = self._calculate_penalty(board, color)
        opponent_penalty = self._calculate_penalty(board, -color)
        return opponent_penalty - player_penalty

    def _calculate_penalty(self, board: Board, color: Color) -> float:
        cells: List[Tuple[int, int]] = board.get_cells_with_color(color)
        penalty = 0
        n, m = board._n, board._m

        for x, y in cells:
            is_corner = (x == 0 or x == m - 1) and (y == 0 or y == n - 1)
            is_edge = x == 0 or x == m - 1 or y == 0 or y == n - 1
            if is_corner:
                penalty += 2
            elif is_edge:
                penalty += 1
        return penalty


class WeightedHeuristic(Heuristic):
    def __init__(self, heuristics: List[Tuple[Heuristic, float]]):
        super().__init__()
        self._heuristics: List[Tuple[Heuristic, float]] = []
        for heuristicCls, weight in heuristics:
            self._heuristics.append((heuristicCls(), weight))

    def _evaluate(self, board: Board, color: Color) -> float:
        total_score = 0
        for heuristic_instance, weight in self._heuristics:
            total_score += heuristic_instance.evaluate(board, color) * weight
        return total_score

    def clear_cache(self):
        super().clear_cache()
        for heuristic_instance, _ in self._heuristics:
            heuristic_instance.clear_cache()

    @staticmethod
    def create_default_heuristic() -> "WeightedHeuristic":
        heuristics = [
            (IsolationHeuristic, 0.5),
            (ControlHeuristic, 0.3),
            (ConnectivityHeuristic, 0.7),
            (EdgePositionHeuristic, 0.2),
        ]
        return WeightedHeuristic(heuristics)


class RandomHeuristic(Heuristic):
    def _evaluate(self, board: Board, color: Color) -> float:
        return random.uniform(-100, 100)


class RLHeuristic(Heuristic):
    def __init__(
        self, heuristics: List[Heuristic], initial_weights: List[float] | None = None
    ):
        super().__init__()
        self._heuristic_instances: List[Heuristic] = [H() for H in heuristics]

        if initial_weights is None:
            self._weights = [1.0] * len(heuristics)
        else:
            self._weights = list(initial_weights)

    def _evaluate(self, board: Board, color: Color) -> float:
        total_score = 0.0
        for i, heuristic_instance in enumerate(self._heuristic_instances):
            feature_value = tanh(heuristic_instance.evaluate(board, color))
            total_score += feature_value * self._weights[i]

        return total_score

    def apply_experiences(self, experiences: List[Tuple[Board, Color, float]]):
        self.update_weights(experiences)

    def update_weights(
        self, experiences: List[Tuple[Board, Color, float]], learning_rate: float = 0.01
    ):

        if not experiences:
            return
        self.clear_cache()
        for h_instance in self._heuristic_instances:
            h_instance.clear_cache()

        for board_state, color, final_outcome in experiences:
            predicted_outcome = self._evaluate(board_state, color)
            error = final_outcome - predicted_outcome
            for i, heuristic_instance in enumerate(self._heuristic_instances):
                feature_value = tanh(heuristic_instance._evaluate(board_state, color))
                self._weights[i] += learning_rate * error * feature_value


# Przebieg gry

Gra rozpoczyna się od utworzenia planszy, a następnie wywołania metody `play` w klasie `Game`. W celu rozpoczęcia gry należy podać graczy, którzy będa brać udział w rozgrywce. Następnie wykonywane są naprzemiennie tury graczy, którzy wykonują ruchy na plaszy. Gra kończy się, gdy nie ma więcej możliwych ruchów. Gracz, który nie mógł wykonać ruchu przegrywa, także nie ma możliwości remisu. Została również zaimplementowana metoda umożliwiająca przeprowadzenie turnieju, w którym gracze będą rywalizować ze sobą przez określoną liczbę gier.

In [ ]:
from typing import List, Tuple
from src.enums.color import Color
from src.utils.decorators import timeit
from src.game.game_result import GameResult
from src.enums.game_state import GameState
from src.player.player import Player
from src.game.board import Board


class Game:
    board: Board
    state: GameState = GameState.WAITING
    history = List[Tuple[Board, Color]]

    def __init__(self, n: int = 10, m: int = 10):
        self.board = Board(n, m)
        self.history = []

    def play_one_turn(self, player: Player) -> GameState:
        move = player.choose_move(self.board)
        if not move:
            return GameState.FINISHED
        print(
            f"{player} moves from ({move.x_start},{move.y_start}) to ({move.x_end},{move.y_end}), move score: {move.score}"
        )
        self.board.perform_move(move)
        self.board.print_board()
        return GameState.IN_PROGRESS

    @timeit
    def play(self, player_one: Player, player_two: Player) -> GameResult:
        self.history = []
        current_player: Player = player_one
        while self.state != GameState.FINISHED:
            self.history.append((self.board.get_copy(), current_player.color))
            self.state = self.play_one_turn(current_player)
            current_player = player_one if current_player == player_two else player_two
        winner: Player = current_player
        loser: Player = player_one if current_player == player_two else player_two

        result: GameResult = GameResult(winner, *self.board.calculate_state())
        winner.acknowledge_experiences(self.history, 1)
        loser.acknowledge_experiences(self.history, -1)

        print(result)
        return result

    @timeit
    def play_tournament(
        self, player_one: Player, player_two: Player, rounds: int = 5
    ) -> None:
        points = {player_one: 0, player_two: 0}
        for i in range(rounds):
            print(f"Round {i + 1} of {rounds}")
            result: GameResult = self.play(player_one, player_two)
            points[result.winner] += 1
            self.board.reset()
            self.history.clear()
            self.state = GameState.WAITING
        print("Tournament results:")
        for player, score in points.items():
            print(f"{player}: {score} points")


# Przykład gry gdzie oba gracze wykorzystują tę samą heurystykę (wersja podstawowa - gracz wykonuje optymalny ruch, przeciwnik wykonuje optymalny ruch):

In [2]:
from src.algorithms.minmax import BaseMinmax
from src.algorithms.heuristics import (
    ConnectivityHeuristic,
    IsolationHeuristic,
)
from src.player.player import Player
from src.game.game import Game

if __name__ == "__main__":

    game: Game = Game(8, 8)
    player_white, player_black = Player.create_default_players(depth=3)
    game.play(player_white, player_black)


Player W moves from (7,7) to (6,7), move score: 0
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W W _
Player B moves from (4,7) to (5,7), move score: 0
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W _ B W _
Player W moves from (6,7) to (5,7), move score: 1
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W _ W _ _
Player B moves from (2,7) to (3,7), move score: 0
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W _ B _ W _ _
Player W moves from (5,7) to (5,6), move score: 1
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
W B W B W W W B
B W _ B _ _ _ _
Player B moves from (0,7) to (0,6), move score: 0
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W 

# Przykład gry z różnymi heurystykami

In [4]:
from src.algorithms.minmax import BaseMinmax
from src.algorithms.heuristics import (
    ControlHeuristic,
    EdgePositionHeuristic,
)
from src.player.player import Player
from src.game.game import Game

if __name__ == "__main__":

    game: Game = Game(8, 8)
    player_white, player_black = Player.create_default_players(depth=3)
    player_black.current_heuristic = EdgePositionHeuristic()
    player_white.current_heuristic = ControlHeuristic()
    game.play(player_white, player_black)


Player W moves from (4,2) to (4,3), move score: 1.0476190476190497
W B W B W B W B
B W B W B W B W
W B W B _ B W B
B W B W W W B W
W B W B W B W B
B W B W B W B W
W B W B W B W B
B W B W B W B W
Player B moves from (6,7) to (6,6), move score: 1
W B W B W B W B
B W B W B W B W
W B W B _ B W B
B W B W W W B W
W B W B W B W B
B W B W B W B W
W B W B W B B B
B W B W B W _ W
Player W moves from (3,5) to (3,4), move score: 1.6174196174196194
W B W B W B W B
B W B W B W B W
W B W B _ B W B
B W B W W W B W
W B W W W B W B
B W B _ B W B W
W B W B W B B B
B W B W B W _ W
Player B moves from (4,7) to (4,6), move score: 2
W B W B W B W B
B W B W B W B W
W B W B _ B W B
B W B W W W B W
W B W W W B W B
B W B _ B W B W
W B W B B B B B
B W B W _ W _ W
Player W moves from (6,4) to (5,4), move score: 1.8629222629222641
W B W B W B W B
B W B W B W B W
W B W B _ B W B
B W B W W W B W
W B W W W W _ B
B W B _ B W B W
W B W B B B B B
B W B W _ W _ W
Player B moves from (2,7) to (2,6), move score: 3
W B W B W

# Przykład turnieju z wykorzystaniem odmiennych heurystyk

In [ ]:
from src.algorithms.minmax import BaseMinmax
from src.algorithms.heuristics import (
    ConnectivityHeuristic,
    ControlHeuristic,
    IsolationHeuristic,
    RLHeuristic,
)
from src.player.player import Player
from src.game.game import Game

if __name__ == "__main__":

    game: Game = Game(5, 5)
    player_white, player_black = Player.create_default_players(depth=3)
    player_black.current_heuristic = ConnectivityHeuristic()
    player_white.current_heuristic = ControlHeuristic()
    game.play_tournament(
        player_white,
        player_black,
        10,
    )


# Przykład turnieju z wykorzystaniem RLHeuristic
### W przypadku turnieju zmniejszono rozmiar planszy do 5x5. Zmiejszenie rozmiaru plaszy pozwala zaooszczędzić czas potrzebny na naukę wag przez alogrytm.

In [ ]:
from src.algorithms.heuristics import (
    ConnectivityHeuristic,
    ControlHeuristic,
    IsolationHeuristic,
    RLHeuristic,
)
from src.player.player import Player
from src.game.game import Game

if __name__ == "__main__":

    game: Game = Game(5, 5)
    player_white, player_black = Player.create_default_players(depth=3)
    player_black.current_heuristic = ConnectivityHeuristic()
    player_white.current_heuristic = RLHeuristic(
        heuristics=[
            ConnectivityHeuristic,
            IsolationHeuristic,
            ControlHeuristic,
        ]
    )
    game.play_tournament(
        player_white,
        player_black,
        250,
    )


Round 1 of 250
Player W moves from (2,4) to (2,3), move score: 2.6343297765012874
W B W B W
B W B W B
W B W B W
B W W W B
W B _ B W
Player B moves from (3,4) to (3,3), move score: 3
W B W B W
B W B W B
W B W B W
B W W B B
W B _ _ W
Player W moves from (0,4) to (0,3), move score: 2.406581150531617
W B W B W
B W B W B
W B W B W
W W W B B
_ B _ _ W
Player B moves from (1,2) to (1,3), move score: 5
W B W B W
B W B W B
W _ W B W
W B W B B
_ B _ _ W
Player W moves from (4,2) to (3,2), move score: 2.67258744543119
W B W B W
B W B W B
W _ W W _
W B W B B
_ B _ _ W
Player B moves from (3,3) to (3,2), move score: -1
W B W B W
B W B W B
W _ W B _
W B W _ B
_ B _ _ W
Player W moves from (3,1) to (2,1), move score: 2.5038820038545198
W B W B W
B W W _ B
W _ W B _
W B W _ B
_ B _ _ W
Player B moves from (0,1) to (1,1), move score: -5
W B W B W
_ B W _ B
W _ W B _
W B W _ B
_ B _ _ W
Player W moves from (2,1) to (1,1), move score: 2.3142552407419053
W B W B W
_ W _ _ B
W _ W B _
W B W _ B
_ B _ _ W
P

# Wykorzystane biblioteki

W celu wykonania projektu, nie wykorzystano żadnych zewnętrznych bibliotek oproćż `pytest` pozwalającej na przeprowadzanie testów kodu i funkcjonalności.